In [ ]:
from paddleocr import PaddleOCRVL

pipeline = PaddleOCRVL()
pipeline.predict()

In [1]:
import re, os , json
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
from pathlib import Path
import joblib
# model_path = r"8_SEGMENT.pkl"
# segment_model = joblib.load(model_path)

# def classify_particular(text):
#     pred = segment_model.predict([text])[0]
#     conf = segment_model.predict_proba([text])[0].max()
#     return pred, round(conf, 4)

AUDIT_RE = re.compile(r"\b(un\-?audited|audited)\b", re.I)
PARTICULARS_RE = re.compile(r"[A-Za-z\s]{4,}")
NUM_LINE_RE = re.compile(r"^\s*[(\[{]?\s*[₹$€£]?\s*[-+]?\s*\d[\d,]*(?:\.\d+)?%?\s*[)\]}]?\s*$")

PREFIX_RE = re.compile(
    r"^\s*(?:\(?[ivxlcdm]+\)|\(?[a-zA-Z]\)|[a-zA-Z][.)]|\(?\d+[.)]?|\([0-9]\))",
    re.I,
)

PREFIX_RE = re.compile(
    r'^\s*(?:\(\d+\)|\([A-Za-z]+\)|\d+[.)]?|[A-Za-z]+[.)])\s+'
)

QT_RE = re.compile(r"\b(?:quarter|quarter ended|3[-\s]?months?(?:\s+ended)?|three[-\s]?months?(?:\s+ended)?)\b",re.I)
FY_RE = re.compile(r"\b(?:year(?:\s+ended)?|12[-\s]?months?(?:\s+ended)?|twelve[-\s]?months?(?:\s+ended)?)\b",re.I)
DATE_RE = re.compile(
    r"""(
        \d{1,2}[./-]\d{1,2}[./-]\d{2,4}|
        \d{1,2}[./-](?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)[./-]\d{2,4}|
        \d{1,2}\s+(?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)\s+\d{2,4}|
        (?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)\s+\d{2,4}
    )""", re.I | re.X,
)


NOTE_RE = re.compile(r"^\s*notes?\b\s*:?", re.I)

In [2]:
def normalize_financial_dataframe(df, top_fraction=0.50):
    df = df.copy().fillna("").map(lambda x: " ".join(str(x).split()))
    metadata = {
        j: {
            "column_type": None,
            "audit": "",
            "date": "",
            "period_type": "",
            "period": "",
        }
        for j in range(df.shape[1])
    }

    start = int(len(df) * top_fraction)

    for j in range(df.shape[1]):
        values = df.iloc[start:, j].astype(str).str.strip()
        non_empty = values[values != ""]

        numeric_ratio = non_empty.map(lambda x: bool(NUM_LINE_RE.fullmatch(x))).mean()
        particulars_ratio = non_empty.map(lambda x: bool(PARTICULARS_RE.findall(x))).mean()

        if numeric_ratio >= 0.4:
            column_type = "NUMERIC"
        elif particulars_ratio >= 0.4:
            column_type = "PARTICULARS"
        else:
            column_type = "OTHER"

        metadata[j]["column_type"] = column_type

    particular_cols = [j for j in metadata if metadata[j]["column_type"] == "PARTICULARS"]
    particular_col = particular_cols[-1] if particular_cols else None

    if particular_col is not None:
        noise_rows, kept_rows = [], []

        for i in range(len(df)):
            row = df.iloc[i]
            values = row.iloc[particular_col:].astype(str).str.strip()
            non_empty = values[values != ""]

            if len(non_empty) > 1 and non_empty.nunique() == 1:
                new_row = [""] * df.shape[1]
                new_row[0] = non_empty.iloc[0]
                noise_rows.append(new_row)
            else:
                kept_rows.append(row.tolist())

        df = pd.DataFrame(kept_rows + noise_rows, columns=df.columns).reset_index(drop=True)
        start = int(len(df) * top_fraction)

    for j in range(df.shape[1]):
        for i in range(start):
            match = AUDIT_RE.findall(str(df.iloc[i, j]).strip())
            if match:
                audit = match[0].lower()
                metadata[j]["audit"] = "Unaudited" if "unaudit" in audit.replace("-", "") else "Audited"
                break

    for j in range(df.shape[1]):
        for i in range(start):
            match = DATE_RE.findall(str(df.iloc[i, j]).strip())
            if match:
                metadata[j]["date"] = match[0]
                break

    for i in range(start):
        for j in range(df.shape[1]):
            text = str(df.iloc[i, j]).strip()
            if QT_RE.findall(text):
                metadata[j]["period_type"] = "QT"
            elif FY_RE.findall(text):
                metadata[j]["period_type"] = "FY"

    numeric_cols = [j for j in metadata if metadata[j]["column_type"] == "NUMERIC"]

    df["ROW_TYPE"] = ""
    df["PREFIX"] = ""

    if particular_col is not None:
        for i in range(len(df)):
            particular = str(df.iloc[i, particular_col]).strip()
            if not particular:
                continue

            values = df.iloc[i, numeric_cols].astype(str).str.strip()
            df.loc[i, "ROW_TYPE"] = "NO DATA" if values.eq("").all() else "DATA"

            match = PREFIX_RE.match(particular)
            if match:
                df.loc[i, "PREFIX"] = match.group()
                #replace the prefix in original
                df.iloc[i, particular_col] = re.sub(PREFIX_RE,"",particular).strip()
                

    move_indices = set()

    for i in range(start):
        for j in range(df.shape[1] - 2):
            text = str(df.iloc[i, j + 2]).strip()
            if AUDIT_RE.findall(text) or DATE_RE.findall(text) or QT_RE.findall(text) or FY_RE.findall(text):
                move_indices.add(i)
                break
        
    moved_rows = df.iloc[sorted(move_indices)].copy()
    df = df.drop(index=move_indices)

    df = df[
        ["ROW_TYPE", "PREFIX"] +
        [col for col in df.columns if col not in ["ROW_TYPE", "PREFIX"]]
    ]

    separator = pd.DataFrame([["="] * df.shape[1]] * 2, columns=df.columns)
    df = pd.concat([df, separator, moved_rows], ignore_index=True)

    metadata_df = pd.DataFrame(
        [
            ["", ""] + [metadata[j]["column_type"] for j in metadata],
            ["", ""] + [metadata[j]["audit"] for j in metadata],
            ["", ""] + [metadata[j]["date"] for j in metadata],
            ["", ""] + [metadata[j]["period_type"] for j in metadata],
        ],
        columns=df.columns
    )

    df = pd.concat([metadata_df, df], ignore_index=True)

    return df, metadata


In [ ]:
from pathlib import Path
from io import StringIO
import json
import pandas as pd
import re


BASE_HTML = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
body { font-family: Arial, sans-serif; }
.source { font-weight: bold; margin-top: 20px; }
.figure-title { font-weight: bold; margin: 10px 0; }
table { border-collapse: collapse; margin: 10px 0; }
td, th { border: 1px solid #999; padding: 4px 8px; }
</style>
</head>
<body>
"""


def clean_table(content):
    soup = BeautifulSoup(content, "html.parser")

    for cell in soup.find_all(["td", "th"]):
        text = cell.get_text(" ", strip=True)
        cell.clear()
        cell.append(text)

    return str(soup)


def process_json(json_file):

    print(f"Processing: {json_file.name}")

    all_dfs = {}
    html = BASE_HTML

    html_out = JSON_DIR / f"{json_file.stem}.html"
    xls_out = JSON_DIR / f"{json_file.stem}.xlsx"

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for page_data in data["pages"]:

        pdf_name = data["batch"]
        page_num = page_data["page"]

        current_title = None
        page_has_table = False
        table_num = 0

        for block in page_data["blocks"]:

            label = str(block.get("label", "")).lower().strip()
            content = block.get("content", "")

            if label == "figure_title":
                current_title = content
                continue

            if label != "table":
                continue

            table_num += 1

            if not page_has_table:
                html += (
                    f'<div class="source">'
                    f"{pdf_name} | Page {page_num}"
                    f"</div>"
                )
                page_has_table = True

            if current_title:
                html += (
                    f'<div class="figure-title">'
                    f"{current_title}"
                    f"</div>"
                )
                current_title = None

            table = clean_table(content)
            html += f"<div>{table}</div>"

            try:
                tables = pd.read_html(StringIO(table))

                if not tables:
                    print(
                        f"  No readable table found "
                        f"on page {page_num}, table {table_num}"
                    )
                    continue

                df = tables[0]

            except (ValueError, ImportError) as exc:
                print(
                    f"  Failed to parse table "
                    f"on page {page_num}, table {table_num}: {exc}"
                )
                continue

            df = (
                df.fillna("")
                .map(lambda x: " ".join(str(x).split()))
            )

            normalized_df, metadata = normalize_financial_dataframe(df)

            sheet_name = f"page{page_num}_table_{table_num}"
            all_dfs[sheet_name] = normalized_df

    html += "</body></html>"

    html_out.write_text(html, encoding="utf-8")
    print(f"Saved HTML: {html_out}")

    if not all_dfs:
        print(f"No tables found in {json_file.name}; Excel file not created.")
        return

    with pd.ExcelWriter(xls_out, engine="openpyxl") as writer:

        for sheet_name, df in all_dfs.items():

            df.to_excel(
                writer,
                sheet_name=sheet_name[:31],
                index=False,
            )

    print(f"Saved Excel: {xls_out}")


for json_file in sorted(JSON_DIR.glob("*.json")):
    process_json(json_file)

print("Done.")